In [3]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold , cross_val_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from sklearn.decomposition import PCA

In [4]:
df = pd.read_csv('gurgaon_properties_post_feature_selection_v2.csv').drop(columns =['store room', 'floor_category','balcony'])

In [5]:
df.columns

Index(['property_type', 'sector', 'price', 'bedRoom', 'bathroom',
       'agePossession', 'built_up_area', 'servant room', 'furnishing_type',
       'luxury_category'],
      dtype='object')

In [6]:
df

,property_type,sector,price,bedRoom,bathroom,agePossession,built_up_area,servant room,furnishing_type,luxury_category
0,flat,sector 68,1.90,3,3,Under Construction,2288.0,1,0,Low
1,flat,sector 84,2.28,4,4,Relatively New,2998.0,1,0,Medium
2,flat,sector 68,1.10,2,2,Relatively New,1150.0,0,1,Medium
3,house,sector 67,8.00,6,7,Moderately Old,3600.0,1,1,High
4,flat,sector 56,1.54,3,4,Old Property,1900.0,1,0,Medium
...,...,...,...,...,...,...,...,...,...,...
3539,house,sector 48,5.15,5,7,Moderately Old,2727.0,1,1,Medium
3540,house,sector 39,1.70,2,2,Moderately Old,900.0,1,0,Low
3541,flat,sector 89,1.85,3,3,New Property,1946.0,1,0,Low
3542,flat,sector 2,0.56,1,2,Moderately Old,723.0,0,0,Medium


In [7]:
# 0 --> unfurnished
# 1 --> semifurnished
# 2 --> furnished

In [8]:
# Numerical = bedRoom, bathroom, built_up_area, sevant room
# Ordinal = property_type, furnishing_type, Luxury_category
# OHE = sector, agePossession

In [9]:
df['agePossession'] = df['agePossession'].replace({
    'Relatively New' : 'new',
    'Moderately Old' : 'old',
    'New Property' : 'new',
    'Old Property' : 'old',
    'Under Construction' : 'under construction'
})

In [10]:
df.head()

,property_type,sector,price,bedRoom,bathroom,agePossession,built_up_area,servant room,furnishing_type,luxury_category
0,flat,sector 68,1.90,3,3,under construction,2288.0,1,0,Low
1,flat,sector 84,2.28,4,4,new,2998.0,1,0,Medium
2,flat,sector 68,1.10,2,2,new,1150.0,0,1,Medium
3,house,sector 67,8.00,6,7,old,3600.0,1,1,High
4,flat,sector 56,1.54,3,4,old,1900.0,1,0,Medium


In [11]:
df['property_type'] = df['property_type'].replace({'flat':0,'house':1})

C:\Users\Mr\AppData\Local\Temp\ipykernel_15440\71934247.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['property_type'] = df['property_type'].replace({'flat':0,'house':1})


In [12]:
df.head()

,property_type,sector,price,bedRoom,bathroom,agePossession,built_up_area,servant room,furnishing_type,luxury_category
0,0,sector 68,1.90,3,3,under construction,2288.0,1,0,Low
1,0,sector 84,2.28,4,4,new,2998.0,1,0,Medium
2,0,sector 68,1.10,2,2,new,1150.0,0,1,Medium
3,1,sector 67,8.00,6,7,old,3600.0,1,1,High
4,0,sector 56,1.54,3,4,old,1900.0,1,0,Medium


In [13]:
df['luxury_category'] = df['luxury_category'].replace({'Low':0,'Medium':1,'High':2})

C:\Users\Mr\AppData\Local\Temp\ipykernel_15440\2576796079.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['luxury_category'] = df['luxury_category'].replace({'Low':0,'Medium':1,'High':2})


In [14]:
df.head()

,property_type,sector,price,bedRoom,bathroom,agePossession,built_up_area,servant room,furnishing_type,luxury_category
0,0,sector 68,1.90,3,3,under construction,2288.0,1,0,0
1,0,sector 84,2.28,4,4,new,2998.0,1,0,1
2,0,sector 68,1.10,2,2,new,1150.0,0,1,1
3,1,sector 67,8.00,6,7,old,3600.0,1,1,2
4,0,sector 56,1.54,3,4,old,1900.0,1,0,1


In [15]:
new_df = pd.get_dummies(df,columns=['sector','agePossession'], drop_first=True)

In [16]:
x = new_df.drop(columns=['price'])
y = new_df['price']

In [17]:
y_log = np.log1p(y)

In [18]:
y_log

0       1.064711
1       1.187843
2       0.741937
3       2.197225
4       0.932164
          ...   
3539    1.816452
3540    0.993252
3541    1.047319
3542    0.444686
3543    2.104134
Name: price, Length: 3544, dtype: float64

In [19]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(x)

In [20]:
X_scaled = pd.DataFrame(X_scaled, columns=x.columns)

In [21]:
X_scaled

,property_type,bedRoom,bathroom,built_up_area,servant room,furnishing_type,luxury_category,sector_gwal pahari,sector_manesar,sector_new,...,sector_sector 92,sector_sector 93,sector_sector 95,sector_sector 99,sector_sector 99a,sector_sector 9a,sector_sohna road,sector_sohna road road,agePossession_old,agePossession_under construction
0,-0.514598,-0.068828,-0.180233,0.374772,1.339758,-0.558784,-0.981828,-0.071449,-0.093938,-0.033615,...,-0.169521,-0.050458,-0.125554,-0.058288,-0.092397,-0.053194,-0.210958,-0.053194,-0.603429,3.441014
1,-0.514598,0.744264,0.515568,0.965597,1.339758,-0.558784,0.442486,-0.071449,-0.093938,-0.033615,...,-0.169521,-0.050458,-0.125554,-0.058288,-0.092397,-0.053194,-0.210958,-0.053194,-0.603429,-0.290612
2,-0.514598,-0.881921,-0.876034,-0.572213,-0.746403,1.220489,0.442486,-0.071449,-0.093938,-0.033615,...,-0.169521,-0.050458,-0.125554,-0.058288,-0.092397,-0.053194,-0.210958,-0.053194,-0.603429,-0.290612
3,1.943265,2.370449,2.602971,1.466550,1.339758,1.220489,1.866800,-0.071449,-0.093938,-0.033615,...,-0.169521,-0.050458,-0.125554,-0.058288,-0.092397,-0.053194,-0.210958,-0.053194,1.657196,-0.290612
4,-0.514598,-0.068828,0.515568,0.051898,1.339758,-0.558784,0.442486,-0.071449,-0.093938,-0.033615,...,-0.169521,-0.050458,-0.125554,-0.058288,-0.092397,-0.053194,-0.210958,-0.053194,1.657196,-0.290612
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3539,1.943265,1.557357,2.602971,0.740085,1.339758,1.220489,0.442486,-0.071449,-0.093938,-0.033615,...,-0.169521,-0.050458,-0.125554,-0.058288,-0.092397,-0.053194,-0.210958,-0.053194,1.657196,-0.290612
3540,1.943265,-0.881921,-0.876034,-0.780250,1.339758,-0.558784,-0.981828,-0.071449,-0.093938,-0.033615,...,-0.169521,-0.050458,-0.125554,-0.058288,-0.092397,-0.053194,-0.210958,-0.053194,1.657196,-0.290612
3541,-0.514598,-0.068828,-0.180233,0.090177,1.339758,-0.558784,-0.981828,-0.071449,-0.093938,-0.033615,...,-0.169521,-0.050458,-0.125554,-0.058288,-0.092397,-0.053194,-0.210958,-0.053194,-0.603429,-0.290612
3542,-0.514598,-1.695013,-0.876034,-0.927540,-0.746403,-0.558784,0.442486,-0.071449,-0.093938,-0.033615,...,-0.169521,-0.050458,-0.125554,-0.058288,-0.092397,-0.053194,-0.210958,-0.053194,1.657196,-0.290612


In [22]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(LinearRegression(), X_scaled, y_log, cv=kfold, scoring='r2')

In [23]:
scores.mean(),scores.std()

(0.8505181753677478, 0.02100049332328092)

In [24]:
lr = LinearRegression()
ridge = Ridge(alpha=0.0001)

In [25]:
lr.fit(X_scaled,y_log)

LinearRegression()

In [26]:
ridge.fit(X_scaled,y_log)

Ridge(alpha=0.0001)

In [29]:
coef_df = pd.DataFrame(ridge.coef_.reshape(1,123), columns=x.columns).stack().reset_index().drop(columns=['level_0']).rename(columns={'level_1':'feature',0:'coef'})

In [30]:
coef_df

,feature,coef
0,property_type,0.126121
1,bedRoom,0.055983
2,bathroom,0.064933
3,built_up_area,0.207906
4,servant room,0.048618
...,...,...
118,sector_sector 9a,-0.005100
119,sector_sohna road,-0.026772
120,sector_sohna road road,-0.011944
121,agePossession_old,-0.008544


In [31]:
# 1. Import necessary libraries
import statsmodels.api as sm

# 2. Add a constant to X
X_with_const = sm.add_constant(X_scaled)


# 3. Fit the model
model = sm.OLS(y_log, X_with_const).fit()

# 4. Obtain summary statistics
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.866
Model:                            OLS   Adj. R-squared:                  0.861
Method:                 Least Squares   F-statistic:                     179.6
Date:                Wed, 01 Apr 2026   Prob (F-statistic):               0.00
Time:                        22:21:32   Log-Likelihood:                 618.73
No. Observations:                3544   AIC:                            -989.5
Df Residuals:                    3420   BIC:                            -224.0
Df Model:                         123                                         
Covariance Type:            nonrobust                                         
                                        coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
const 

In [32]:
y_log.std()

0.5551372282244343

In [33]:
X_scaled['bedRoom'].std()

1.0001411133853069

In [34]:
0.21 * (0.557/1)

0.11697

In [35]:
np.expm1(0.030)

0.030454533953516858

In [36]:
2.4726962617564903e-05 * 100

0.0024726962617564905